# AIC 2026 — Multi-Modal Keyframe-Level Vector Generation (SigLIP-SO400M + BGE-m3)

> **Recommended Kaggle Accelerator**: **GPU T4 x 2** or **P100**.

Generates **one row per keyframe** with rich contextual metadata (`media_info` title/description/keywords + shot captions).

Outputs:
1. **Visual Vectors (1152-d)**: `google/siglip-so400m-patch14-384` per keyframe image.
2. **Dense Text Vectors (1024-d)**: `BAAI/bge-m3` — combined `Title | Description | Caption`.
3. **`metadata_df.pkl`**: DataFrame with `video_name`, `n`, `frame_idx`, `pts_time`, `shot_id`, `video_title`, `video_description`, `video_keywords`, `caption`.
4. **`bm25_corpus.pkl`**: Text corpus aligned 1:1 with embeddings.
5. **Future/Experimental**: Optional focal entity weighting flag (`ENABLE_FOCAL_ENTITY_WEIGHTING = False`, see ADR 0005).

In [ ]:
!pip install -q transformers sentence-transformers pandas numpy tqdm torch torchvision Pillow


In [ ]:
import os
import re
import json
import glob
import pickle
import shutil
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from transformers import AutoProcessor, AutoModel
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

# 1. Hardware & Path Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Compute device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

OUTPUT_DIR = Path('/kaggle/working/kaggle_output')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Future / Experimental Flag (ADR 0005 - Focal Entity Weighting)
ENABLE_FOCAL_ENTITY_WEIGHTING = False

# 2. Auto-Discover Input Folders across /kaggle/input
print('Discovering input directories...')
captions_files = sorted(glob.glob('/kaggle/input/**/captions/*.json', recursive=True))
shots_files = sorted(glob.glob('/kaggle/input/**/shot_boundaries/*.json', recursive=True))
media_files = sorted(glob.glob('/kaggle/input/**/media_info/*.json', recursive=True))
image_dirs = sorted(
    glob.glob('/kaggle/input/**/keyframe_images', recursive=True) +
    glob.glob('/kaggle/input/**/extracted_keyframe_images', recursive=True)
)

print(f'Found {len(captions_files)} caption JSONs, {len(shots_files)} shot JSONs, {len(media_files)} media_info JSONs, {len(image_dirs)} image roots.')

caption_map = {Path(p).stem: p for p in captions_files}
shots_map = {Path(p).stem: p for p in shots_files}
media_map = {Path(p).stem: p for p in media_files}

# Helper: Clean description boilerplate
def clean_description(desc: str, max_chars: int = 250) -> str:
    if not desc or not isinstance(desc, str):
        return ''
    # Remove URLs
    text = re.sub(r'https?:\S+|bit\.ly/\S+', '', desc)
    # Remove hashtags and subscriber/social callouts
    cleaned_lines = []
    for line in text.splitlines():
        l = line.strip()
        if not l or l.startswith('#') or l.startswith('👉') or l.startswith('►') or l.startswith('📣'):
            continue
        if any(w in l.lower() for w in ['subscribe', 'bản quyền', 'fanpage', 'liên hệ', 'facebook.com']):
            continue
        cleaned_lines.append(l)
    summary = ' '.join(cleaned_lines)
    summary = re.sub(r'\s+', ' ', summary).strip()
    return summary[:max_chars].strip()

# Helper: Parse focal entities (ADR 0005)
def extract_focal_entities(memory_text: str) -> list:
    if not memory_text or memory_text == 'None':
        return []
    focal = []
    for line in memory_text.splitlines():
        clean = line.strip()
        if '(previous focus)' in clean.lower() or '(seen earlier)' in clean.lower():
            continue
        clean = re.sub(r'\((?:new|current) focus\)', '', clean, flags=re.IGNORECASE)
        clean = re.sub(r'^[\s\-\*	0-9\.\)]+', '', clean).strip()
        if clean:
            focal.append(clean)
    return focal

def find_keyframe_image(vid: str, img_name: str) -> str:
    for img_root in image_dirs:
        cand = Path(img_root) / vid / img_name
        if cand.exists():
            return str(cand)
    return None

# 3. Build Keyframe-Level Dataset (one row per keyframe)
print('Building keyframe-level dataset with media_info metadata...')
records = []
all_vids = sorted(set(list(caption_map.keys()) + list(shots_map.keys()) + list(media_map.keys())))

for vid in all_vids:
    # Load media info
    vid_title = ''
    vid_desc = ''
    vid_kws = ''
    if vid in media_map:
        try:
            with open(media_map[vid], 'r', encoding='utf-8') as f:
                m_info = json.load(f)
                vid_title = m_info.get('title', '')
                raw_desc = m_info.get('description', '')
                vid_desc = clean_description(raw_desc)
                kws = m_info.get('keywords', [])
                vid_kws = ', '.join(kws) if isinstance(kws, list) else str(kws)
        except Exception:
            pass

    # Load shot boundaries
    shot_data = {}
    if vid in shots_map:
        try:
            with open(shots_map[vid], 'r', encoding='utf-8') as f:
                for s in json.load(f):
                    shot_data[s.get('shot_id', 0)] = s
        except Exception:
            pass

    # Load captions
    cap_data = {}
    if vid in caption_map:
        try:
            with open(caption_map[vid], 'r', encoding='utf-8') as f:
                for c in json.load(f):
                    cap_data[c.get('shot_id', 0)] = c
        except Exception:
            pass

    # Iterate over every keyframe in every shot
    all_shot_ids = sorted(set(list(shot_data.keys()) + list(cap_data.keys())))
    for shot_id in all_shot_ids:
        sb = shot_data.get(shot_id, {})
        cap = cap_data.get(shot_id, {})
        caption_text = cap.get('caption', '')
        memory_text = cap.get('memory', '')
        start_time = sb.get('start_time', cap.get('start_time', 0.0))
        end_time = sb.get('end_time', cap.get('end_time', 0.0))

        keyframes = sb.get('keyframes', [])
        if not keyframes:
            records.append({
                'video_name': vid,
                'n': 0,
                'frame_idx': 0,
                'pts_time': (start_time + end_time) / 2.0,
                'fps': 30.0,
                'shot_id': shot_id,
                'start_time': start_time,
                'end_time': end_time,
                'video_title': vid_title,
                'video_description': vid_desc,
                'video_keywords': vid_kws,
                'caption': caption_text,
                'memory': memory_text,
                'image_path': None
            })
            continue

        for kf in keyframes:
            img_name = kf.get('image', '')
            img_path = find_keyframe_image(vid, img_name) if img_name else None
            records.append({
                'video_name': vid,
                'n': kf.get('n', 0),
                'frame_idx': kf.get('frame_idx', 0),
                'pts_time': kf.get('pts_time', 0.0),
                'fps': kf.get('fps', 30.0),
                'shot_id': shot_id,
                'start_time': start_time,
                'end_time': end_time,
                'video_title': vid_title,
                'video_description': vid_desc,
                'video_keywords': vid_kws,
                'caption': caption_text,
                'memory': memory_text,
                'image_path': img_path
            })

df_meta = pd.DataFrame(records)
print(f'Total keyframe records: {len(df_meta)}')
print(f'Unique videos: {df_meta["video_name"].nunique()}')
found_imgs = df_meta['image_path'].notnull().sum()
print(f'Matched keyframe images: {found_imgs}/{len(df_meta)}')

# 4. Load Models on GPU
print('\nLoading SigLIP-SO400M...')
siglip_model_name = 'google/siglip-so400m-patch14-384'
siglip_processor = AutoProcessor.from_pretrained(siglip_model_name)
siglip_model = AutoModel.from_pretrained(siglip_model_name).to(device).eval()

print('Loading BGE-m3...')
bge_model = SentenceTransformer('BAAI/bge-m3', device=device)

# 5. Generate Text Embeddings (BGE-m3, 1024-d) with Title, Description, and Caption
text_corpus = []
for _, row in df_meta.iterrows():
    parts = []
    if row['video_title']:
        parts.append(f"Title: {row['video_title']}")
    if row['video_description']:
        parts.append(f"Description: {row['video_description']}")
    if row['caption']:
        parts.append(f"Caption: {row['caption']}")
        
    if ENABLE_FOCAL_ENTITY_WEIGHTING:
        focal_ents = extract_focal_entities(row['memory'])
        if focal_ents:
            parts.append(f"[CURRENT FOCUS: {' '.join(focal_ents)}]")
            
    combined_text = ' | '.join(parts) if parts else 'Video keyframe'
    text_corpus.append(combined_text)

print(f'Encoding {len(text_corpus)} text embeddings with BGE-m3...')
text_embeddings = bge_model.encode(text_corpus, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
np.save(OUTPUT_DIR / 'text_embeddings.npy', text_embeddings.astype(np.float32))
print(f'Saved text_embeddings.npy: shape {text_embeddings.shape}')

# 6. Generate SigLIP-SO400M Visual Embeddings (1152-d) — per keyframe
print(f'Encoding {len(df_meta)} visual embeddings with SigLIP-SO400M...')
vis_embeddings = np.zeros((len(df_meta), 1152), dtype=np.float32)
VIS_BATCH = 64

for i in tqdm(range(0, len(df_meta), VIS_BATCH), desc='SigLIP Visual'):
    batch_rows = df_meta.iloc[i : i + VIS_BATCH]
    batch_imgs = []
    valid_indices = []

    for idx_offset, (_, row) in enumerate(batch_rows.iterrows()):
        img_p = row['image_path']
        if img_p and os.path.exists(img_p):
            try:
                img = Image.open(img_p).convert('RGB')
                batch_imgs.append(img)
                valid_indices.append(i + idx_offset)
            except Exception:
                pass

    if batch_imgs:
        inputs = siglip_processor(images=batch_imgs, return_tensors='pt').to(device)
        with torch.no_grad():
            out = siglip_model.get_image_features(**inputs)
            if hasattr(out, 'pooler_output') and out.pooler_output is not None:
                img_feats = out.pooler_output
            elif hasattr(out, 'last_hidden_state'):
                img_feats = out.last_hidden_state[:, 0, :]
            elif not isinstance(out, torch.Tensor) and hasattr(out, '__getitem__'):
                img_feats = out[0]
            else:
                img_feats = out

            img_feats = F.normalize(img_feats, p=2, dim=-1)
            for out_idx, slot in enumerate(valid_indices):
                vis_embeddings[slot] = img_feats[out_idx].cpu().numpy().astype(np.float32)

np.save(OUTPUT_DIR / 'visual_embeddings.npy', vis_embeddings)
print(f'Saved visual_embeddings.npy: shape {vis_embeddings.shape}')

# 7. Save Metadata & BM25 Corpus
df_save = df_meta.drop(columns=['image_path'])
df_save.to_pickle(OUTPUT_DIR / 'metadata_df.pkl')
with open(OUTPUT_DIR / 'bm25_corpus.pkl', 'wb') as f:
    pickle.dump({'corpus': [[t] for t in text_corpus]}, f)

print(f'\nSaved metadata_df.pkl: {len(df_save)} rows, columns: {list(df_save.columns)}')

# 8. Zip Artifacts for 1-Click Download
shutil.make_archive('/kaggle/working/kaggle_vectors', 'zip', OUTPUT_DIR)
print('\nExtraction complete!')
print('Download: /kaggle/working/kaggle_vectors.zip')
